## Transformação de dados - produtos_estoques_movimentacoes

#### 1.Carregar tabela do bronze

In [ ]:
import sys
sys.path.append("/app/pipeline_spark/")


from utils import create_spark_session, load_config, save_table
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tabela_nome = "produtos_estoques_movimentacoes"

spark = create_spark_session("produtos_estoques_movimentacoes")
config = load_config()

bronze_path = config["storage"]["bronze"]["path"]
silver_path = config["storage"]["silver"]["path"]


# ler bronze
df = spark.read.format("delta").load(
    f"{bronze_path}/{tabela_nome}"
)

#### 2.Executar transformação

In [ ]:

# transformação
df_silver = (
    df
    .select("saldo", "produto_id", "empresa_id", "data_lancamento", "hora_lancamento", "id")
    .withColumn("data_lancamento", F.to_date("data_lancamento", "yyyy-MM-dd"))
)

#### 3.Armazenar dados na camada silver

In [ ]:
# salvar silver
path = f"{silver_path}/{tabela_nome}"

if DeltaTable.isDeltaTable(spark, path):

    delta_table = DeltaTable.forPath(spark, path)

    (
        delta_table.alias("t")
        .merge(
            df_silver.alias("s"),
            """t.id = s.id"""
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    df_silver.write.format("delta").save(path)

#Manter arquivos antigos por 7 dias (168 horas)
spark.sql(f"VACUUM delta.`{path}` RETAIN 168 HOURS")